# Personalized Financial Advisory System Using AI
Step-by-step process: Import, Load, Preprocess, Exploratory Analysis, and Model Implementation.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

## 2. Load Data

In [ ]:
df = pd.read_csv('personal_finance_tracker_dataset.csv')
print(f"Dataset shape: {df.shape}")
display(df.head())

## 3. Data Preprocessing
Handling missing values, creating new features, and encoding categorical variables.

In [ ]:
# Convert dates
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month

# Handle missing values
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Feature Engineering
df['discretionary_ratio'] = df['discretionary_spending'] / df['monthly_income']
df['essential_ratio'] = df['essential_spending'] / df['monthly_income']
df['savings_to_goal_ratio'] = df['actual_savings'] / df['budget_goal']
df['runway_months'] = df['emergency_fund'] / df['essential_spending']

# Encode Categorical Variables
cat_cols = ['financial_scenario', 'income_type', 'rent_or_mortgage', 'cash_flow_status']
encoders = {}
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

print("Preprocessing Complete.")
display(df[['user_id', 'essential_ratio', 'discretionary_ratio']].head())

## 4. Model Implementation: Smart Investment Recommendations
Using K-Means clustering to segment users based on financial health and risk tolerance.

In [ ]:
# Select features for clustering
features = ['monthly_income', 'actual_savings', 'credit_score', 'savings_rate']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Apply KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
df['investment_cluster'] = kmeans.fit_predict(X_scaled)

def recommend_portfolio(cluster):
    if cluster == 0:
        return "Conservative: 60% Bonds, 30% Large-Cap Stocks, 10% Cash (Low Risk Tolerance)"
    elif cluster == 1:
        return "Balanced: 50% Large-Cap Stocks, 30% Bonds, 20% International Stocks (Medium Risk)"
    else:
        return "Aggressive: 70% Stocks (Tech/Growth), 20% Crypto/Alt, 10% Bonds (High Risk/High Savings)"

df['portfolio_recommendation'] = df['investment_cluster'].apply(recommend_portfolio)
display(df[['user_id', 'monthly_income', 'credit_score', 'portfolio_recommendation']].sample(5))

## 5. Other AI Features: Purchase Timing, Loan Advisory, Behavioral Insights

In [ ]:
# 5.1 Purchase Timing Advisor
def purchase_timing_advisor(item_type, current_month, financial_scenario):
    if item_type.lower() == 'car':
        if current_month in [11, 12] or financial_scenario == 'recession':
            return "Optimal time to buy! Dealerships offer year-end discounts."
        else:
            return "Wait for end-of-year sales (Nov-Dec)."
    return "Neutral time to buy."

df['car_purchase_advice'] = df.apply(lambda row: purchase_timing_advisor('car', row['month'], row['financial_scenario']), axis=1)

# 5.2 Behavioral Insights
avg_discretionary = df['discretionary_spending'].mean()
def behavioral_insight(spending):
    if spending > avg_discretionary * 1.2:
        return f"Warning: Spending (${spending:.2f}) is >20% above average."
    return "Spending habits look healthy."

df['behavioral_insight'] = df['discretionary_spending'].apply(behavioral_insight)

# 5.3 Tax Optimization
def tax_optimization(annual_income):
    if annual_income > 160000:
        return "High Bracket: Maximize 401(k), consider Municipal Bonds."
    elif annual_income > 80000:
        return "Medium Bracket: Maximize HSA, contribute to traditional 401(k)."
    else:
        return "Lower Bracket: Consider Roth IRA while your tax rate is relatively low."

df['annual_income'] = df['monthly_income'] * 12
df['tax_advice'] = df['annual_income'].apply(tax_optimization)
display(df[['user_id', 'car_purchase_advice', 'behavioral_insight', 'tax_advice']].head())

## 6. Explainable AI Dashboard

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 6))

# Left: Credit Score vs Debt to Income Ratio
sns.scatterplot(data=df, x='debt_to_income_ratio', y='credit_score', hue='portfolio_recommendation', ax=axs[0])
axs[0].set_title('Credit Score vs DTI (Colored by Portfolio)')
axs[0].legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

# Right: Discretionary Spending Distribution compared to Mean
sns.kdeplot(data=df, x='discretionary_spending', ax=axs[1], fill=True)
axs[1].axvline(df['discretionary_spending'].mean(), color='r', linestyle='--', label='Average Spending')
axs[1].set_title('Discretionary Spending Distribution')
axs[1].legend()

plt.tight_layout()
plt.show()